## 1. Process dataset

Load and read dataset and transpose it 

In [92]:
import pandas as pd

json_path = "data/games.json"

df_original = pd.read_json(json_path).T

Use a copy for DataFrame instead of touching the original

In [98]:
df = df_original.copy()
df.index.name = "app_id"
df = df.reset_index()
df.head()

,app_id,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,...,positive,negative,estimated_owners,average_playtime_forever,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu,tags
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0,0.0,0,,,,,...,0,0,0 - 0,0,0,0,0,0,0,[]
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0,5.24,0,"Springtime, April: when the cherry trees come ...","Springtime, April: when the cherry trees come ...","Spring has come, and our protagonist, Yukinari...",,...,252,3,0 - 20000,8,0,8,0,65,0,"{'Adventure': 27, 'Visual Novel': 19, 'Anime':..."
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0,4.99,0,"Immerse yourself in the most beloved, mystical...","Immerse yourself in the most beloved, mystical...",Discover an entrancing and spectacular world!,,...,21,3,0 - 20000,0,0,0,0,0,0,"{'Casual': 83, 'Card Game': 52, 'Solitaire': 4..."
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0,8.99,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...","synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",Yuha! I'll start the broadcast! Hakko's extrem...,,...,0,0,0 - 20000,0,0,0,0,0,1,[]
4,3631080,Maze Quest VR,"Apr 24, 2025",0,4.99,0,Its not just a Maze; its a Quest! Enter the ca...,Its not just a Maze; its a Quest! Enter the ca...,Its not just a Maze; its a Quest! Enter the ca...,,...,0,0,0 - 20000,0,0,0,0,0,0,[]


Drop unneeded columns

In [99]:
import re

df = df.dropna(subset=["name"])
df = df[((df["positive"] + df["negative"]) > 0) | (df["recommendations"] > 0)]

metadata_columns = ["categories", "genres", "tags"]
df = df.dropna(subset=metadata_columns, how="all")

df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')

dedup_keys = ["name", "release_date", "detailed_description"]
df["null_count"] = df.isnull().sum(axis=1)

df = (
  df.sort_values(by=["null_count", "app_id"], ascending=[True, True])
  .drop_duplicates(subset=dedup_keys, keep="first")
  .drop(columns=["null_count"])
  .reset_index(drop=True)
)

columns_to_drop = [
  "detailed_description",
  "about_the_game",
  "reviews",
  "header_image",
  "website",
  "support_url",
  "support_email",
  "metacritic_score",
  "metacritic_url",
  "notes",
  "packages",
  "screenshots",
  "movies",
  "user_score",
  "score_rank",
  "estimated_owners",
  "average_playtime_forever",
  "average_playtime_2weeks",
  "median_playtime_forever",
  "median_playtime_2weeks",
  "peak_ccu",
]
df = df.drop(columns=columns_to_drop, errors="ignore")

def clean_text(text):
  text = re.sub(r'[^\w\s-]', '', text)
  text = text.strip(' -')
  text = re.sub(r'[\s-]+', '_', text)
  return text.lower()

def process_metadata(val):
  if isinstance(val, dict):
    raw_list = list(val.keys())
  elif isinstance(val, list):
    raw_list = val
  else:
    return []
  return [clean_text(item) for item in raw_list if str(item).strip()]

df['cleaned_categories'] = df['categories'].map(process_metadata)
df['cleaned_genres'] = df['genres'].map(process_metadata)
df['cleaned_tags'] = df['tags'].map(process_metadata)

df['combined_metadata'] = (
  df['cleaned_categories'] + df['cleaned_genres'] + df['cleaned_tags']
).map(lambda items: ' '.join(set(items)))

df = df.drop(columns=metadata_columns, errors="ignore")

df

,app_id,name,release_date,required_age,price,dlc_count,short_description,windows,mac,linux,...,full_audio_languages,developers,publishers,positive,negative,discount,cleaned_categories,cleaned_genres,cleaned_tags,combined_metadata
0,10,Counter-Strike,2000-11-01,0,1.99,0,Play the world's number 1 online action game. ...,True,True,True,...,"[English, French, German, Italian, Spanish - S...",[Valve],[Valve],243818,6427,80,"[multi_player, pvp, online_pvp, sharedsplit_sc...",[action],"[action, fps, multiplayer, shooter, classic, t...",old_school fps multiplayer nostalgia score_att...
1,20,Team Fortress Classic,1999-04-01,0,1.24,0,One of the most popular online action games of...,True,True,True,...,[],[Valve],[Valve],7602,1136,75,"[multi_player, pvp, online_pvp, sharedsplit_sc...",[action],"[action, fps, multiplayer, classic, hero_shoot...",old_school fps multiplayer mod remake first_pe...
2,30,Day of Defeat,2003-05-01,0,1.24,0,Enlist in an intense brand of Axis vs. Allied ...,True,True,True,...,[],[Valve],[Valve],6414,688,75,"[multi_player, camera_comfort, color_alternati...",[action],"[fps, world_war_ii, multiplayer, shooter, acti...",old_school difficult fps multiplayer war tacti...
3,40,Deathmatch Classic,2001-06-01,0,1.24,0,Enjoy fast-paced multiplayer gaming with Death...,True,True,True,...,[],[Valve],[Valve],2618,545,75,"[multi_player, pvp, online_pvp, sharedsplit_sc...",[action],"[action, fps, classic, multiplayer, shooter, f...",old_school difficult fps multiplayer sci_fi go...
4,50,Half-Life: Opposing Force,1999-11-01,0,1.24,0,Return to the Black Mesa Research Facility as ...,True,True,True,...,[],[Gearbox Software],[Valve],24363,1198,75,"[single_player, multi_player, custom_volume_co...",[action],"[fps, action, classic, sci_fi, singleplayer, s...",playable_without_timed_input adjustable_diffic...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85911,4872540,AeroBird,2026-07-11,0,0.99,0,AeroBird is a fast-paced acrobatic platformer ...,True,False,False,...,[],[POYANA BRADYLUI],[Poyana Bradylui],0,0,0,"[single_player, family_sharing]","[adventure, casual, indie, sports]",[],casual indie single_player adventure family_sh...
85912,4881550,GEL RUN,2026-07-10,0,0.99,0,GEL RUN is a colorful obstacle-course adventur...,True,False,False,...,[],[Kaloyan Stoyanov],[Kaloyan Stoyanov],0,0,0,"[single_player, family_sharing]","[adventure, casual, indie, simulation, sports]",[],casual indie single_player adventure family_sh...
85913,4885620,Ocean Rush,2026-07-12,0,0.59,0,Ocean Rush is a fast-paced ocean escape advent...,True,False,False,...,[],[riko],[riko],0,0,0,"[single_player, family_sharing]","[adventure, casual, indie, sports]",[],casual indie single_player adventure family_sh...
85914,4929970,No Socks RPG,2026-07-27,0,9.99,1,"Fight groovy enemies, and recruit a team of se...",True,True,False,...,[],[Spicyfuse],[cakesin],0,0,0,"[single_player, family_sharing]","[adventure, casual, rpg]",[],casual single_player adventure family_sharing rpg


In [123]:
from sklearn.feature_extraction.text import TfidfVectorizer

catalog_vectorizer = TfidfVectorizer(token_pattern=r'\S+')
catalog_matrix = catalog_vectorizer.fit_transform(df['combined_metadata'])

## 2. Process intents

In [ ]:
import json

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

intents_path = "data/intents.json"

with open(intents_path, 'r') as f:
  data = json.load(f)

patterns, pattern_tags = [], []
for intent in data['intents']:
  for pattern in intent['patterns']:
    patterns.append(pattern.lower())
    pattern_tags.append(intent)

intent_vectorizer = TfidfVectorizer().fit(patterns)
pattern_vectors = intent_vectorizer.transform(patterns)


def get_intent(clean_input, confidence_threshold=0.25):
  input_vector = intent_vectorizer.transform([clean_input])
  similarities = cosine_similarity(input_vector, pattern_vectors)[0]
  best_match = np.argmax(similarities)

  if similarities[best_match] < confidence_threshold:
    return None
  return pattern_tags[best_match]


def search_dataset(user_query, top_n=5):
  formatted_query = re.sub(r'[\s-]+', '_', user_query)
  query_vector = catalog_vectorizer.transform([formatted_query])
  similarities = cosine_similarity(query_vector, catalog_matrix).flatten()
  top_indices = np.argsort(similarities)[-top_n:][::-1]

  results = []
  for idx in top_indices:
    if similarities[idx] > 0.1:
      game = df.iloc[idx]
      price = game['price'] if game['price'] > 0 else "Free"
      results.append(f"{game['name']} - Price: {price} - Release Date: {game['release_date']}")

  if not results:
    return "No matching games found."
  return "Here are the top results found:\n" + "\n".join(results)


def check_platform(clean_input):
  matches = df[df['name'].str.contains(clean_input, case=False, na=False)]

  if matches.empty:
    return "Game not found."

  game = matches.iloc[0]
  supported_platforms = []
  if game['windows']: supported_platforms.append("Windows")
  if game['mac']: supported_platforms.append("Mac")
  if game['linux']: supported_platforms.append("Linux")

  return f"{game['name']} is available on: {', '.join(supported_platforms)}" if supported_platforms else f"{game['name']} is not available on any platform."

def search_by_price(clean_input):
  if "free" in clean_input:
    results = df[df['price'] == 0].sort_values(by='recommendations', ascending=False).head(5)
    return "Top free games:\n" + "\n".join([f"- **{r['name']}**" for _, r in results.iterrows()])

  numbers = re.findall(r'\d+', clean_input)
  if numbers:
    max_price = float(numbers[0])
    results = df[df['price'] <= max_price]
    if not results.empty:
      return results.sort_values(by='recommendations', ascending=False).head(5)

  return search_dataset(clean_input)

def get_top_rated(top_n=5):
  top_games = df.sort_values(by='recommendations', ascending=False).head(top_n)
  return "\n".join([f"{game['name']} - Rating: {game['recommendations']}" for _, game in top_games.iterrows()])

def handle_response(clean_input):
  intent = get_intent(clean_input)

  if not intent:
    return "I'm sorry, I didn't understand that. Could you please rephrase?"

  response = np.random.choice(intent['responses'])

  if response in ["SEARCH_GENRE_TRIGGER", "SEARCH_TITLE_TRIGGER"]:
    return search_dataset(clean_input)
  elif response == "CHECK_PLATFORM_TRIGGER":
    return check_platform(clean_input)
  elif response == "SEARCH_PRICE_TRIGGER":
    return search_by_price(clean_input)
  elif response == "SEARCH_TOP_RATED_TRIGGER":
    return get_top_rated()
  return response

def preprocess_input(text):
  text = text.lower().strip()
  text = re.sub(r'[^\w\s-]', '', text)
  text = re.sub(r'\s+', ' ', text)
  return text

while True:
  raw_input = input("You: ")
  clean_input = preprocess_input(raw_input)

  if clean_input in ['exit', 'quit']:
    print("Bot: Goodbye!")
    break

  if not clean_input:
    continue

  print("You:", clean_input)
  print("Bot:", handle_response(clean_input))

You: info for team fortress 2
Bot: I use content-based machine learning to suggest games tailored to your tastes, filter by budget or platform, and assist with Steam store and refund FAQs.
You: show me team fortress 2
Bot: No matching games found.
You: portal 2
Bot: No matching games found.
You: portal
Bot: No matching games found.
Bot: Goodbye!
